# Spacing Statistics - RingEnergy

## 1. Importing / Installing Packages

In [20]:
import os # Importing os module for operating system dependent functionality

import glob # Importing glob module for file pattern matching

import pandas as pd # Importing pandas package

# Set the maximum number of columns to display to None
pd.set_option('display.max_columns', None)

from pathlib import Path # Importing Path class from pathlib for filesystem path manipulation

from src.utils import reorder_columns, clean_column_names # Importing utility functions from the utils module

from src.well_data import GeoSurveyProcessor, WellDataLoader, WellSpacingCalculator, DirectionalBenchNeighbors, debug_pair_spacing # Importing custom classes for well data processing

from src.utils import DatabricksOdbcConnector # Importing DatabricksOdbcConnector class from utils module

## 2. Importing Data to Dataframes

In [2]:
def load_uwi_files(folder_path: str) -> pd.DataFrame:
    """
    Reads all .UWI files in the given folder into a single DataFrame.
    
    Parameters
    ----------
    folder_path : str
        Path to the folder containing .UWI files.
    
    Returns
    -------
    pd.DataFrame
        DataFrame with columns ['UWI', 'county'].
    """
    all_rows = []

    # Find all .UWI files in the folder
    uwi_paths = glob.glob(os.path.join(folder_path, "*.UWI"))

    for path in uwi_paths:
        county = Path(path).stem  # Extract filename without extension
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            uwis = [line.strip() for line in f if line.strip()]

        print(f"{county}: {len(uwis)} rows")

        for uwi in uwis:
            all_rows.append({"uwi": uwi, "county": county})

    df = pd.DataFrame(all_rows, columns=["uwi", "county"])
    return df

### 2.1 Importing header

In [3]:
data_loader = WellDataLoader(
    db = DatabricksOdbcConnector(),
    log_dir=r"C:\Users\Apoorva.Saxena\OneDrive - Sitio Royalties\Desktop\Project - Apoorva\Python\Parent_Child_Spacing\logs"
    )

In [4]:
df_header = data_loader.get_header_data(
    source=r"C:\Users\Apoorva.Saxena\OneDrive - Sitio Royalties\Desktop\Project - Apoorva\For Matt\Well Header\uwi_by_county_bench.xlsx",
    column_map={
        "uwi": "uwi",
        "api10":"api10",
        "bench": "bench_final",
        "hole_direction": "HoleDirection_Final",
        "ihs_missing": "IHS_Missing",
        "ihs_ds_missing": "IHS_DS_Missing"
    },
    dtype={"uwi": str, "api10": str}
)

[WellDataLoaderLogger] INFO (08-19 02:48 PM): Loading header data from file: C:\Users\Apoorva.Saxena\OneDrive - Sitio Royalties\Desktop\Project - Apoorva\For Matt\Well Header\uwi_by_county_bench.xlsx (Line: 242) [well_data_manager.py]



### 2.2 Getting Directional Surveys

In [5]:
df_directional = data_loader.get_directional_data()

[WellDataLoaderLogger] INFO (08-19 02:48 PM): Loading directional data from SQL. (Line: 311) [well_data_manager.py]

c:\users\apoorva.saxena\onedrive - sitio royalties\desktop\project - apoorva\python\parent_child_spacing\src\utils\database_manager.py:28: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  result_df = pd.read_sql(sql_query, self.connection)


### 2.3 Missing DS from Header

In [6]:
missing_uwi = set(df_header['uwi']) - set(df_directional['uwi'])
print(f"Number of missing UWI: {len(missing_uwi)}")
# list(missing_uwi)

Number of missing UWI: 663


In [7]:
# sorted(set(df_directional['uwi']) - set(df_header['uwi'])) # List of UWI in directional but not in header

In [8]:
df_missing_directional = df_header.merge(df_directional[['uwi']].drop_duplicates(), 
                                         on='uwi', how='left', indicator=True).loc[lambda d: d['_merge'].eq('left_only')].drop(columns=['_merge']).reset_index(drop=True)

## 3. Computing UTM Coordinates

In [9]:
# Initialize the GeoSurveyProcessor with the log directory
geo = GeoSurveyProcessor(
    log_dir=r"C:\Users\Apoorva.Saxena\OneDrive - Sitio Royalties\Desktop\Project - Apoorva\Python\Parent_Child_Spacing\logs")

[GeoLogger] INFO (08-19 02:48 PM): GeoSurveyProcessor initialized. (Line: 477) [well_data_manager.py]



In [10]:
df_utm = geo.compute_utm_coordinates(df=df_directional)

[GeoLogger] INFO (08-19 02:48 PM): ✅ Using lat/lon from input DataFrame. (Line: 653) [well_data_manager.py]

[GeoLogger] INFO (08-19 02:48 PM): ✅ UTM coordinate computation complete in 0.21 sec. (Line: 708) [well_data_manager.py]



In [11]:
# Filter the DataFrame to get only the lateral sections after the heel point
df_utm_lateral = geo.filter_after_heel_point(df=df_utm)

## 4. Calculate Spacing I-K Piars

In [12]:
# Initialize the WellSpacingCalculator with the lateral trajectories DataFrame
spacing_stats_ik = WellSpacingCalculator(trajectories=df_utm_lateral)

In [ ]:
# spacing_stats_ik._calculate_spacing_statistics(
#     batch_size=200_000,
#     max_distance_miles=3.0,
#     save_batches_dir=r"C:\Users\Apoorva.Saxena\OneDrive - Sitio Royalties\Desktop\Project - Apoorva\For Matt\Spacing Stats",
#     step_ft=100,
#     max_crossline_ft=2500
# )

🚀 Calculating Spacing (Parallel): |████████████████████████████████████████| 100% 1/1 [00:00<00:00]


✅ All batches saved to C:\Users\Apoorva.Saxena\OneDrive - Sitio Royalties\Desktop\Project - Apoorva\For Matt\Spacing Stats


In [14]:
df_spacing_ik = spacing_stats_ik._load_saved_batches(
    batch_folder=r"C:\Users\Apoorva.Saxena\OneDrive - Sitio Royalties\Desktop\Project - Apoorva\For Matt\Spacing Stats"
)

🔍 Found 1 batch files. Loading and combining...
✅ Loaded 88,944 rows from all batches.


## 5. Get OneLine

### 5.1 Joining bench with spacing ik dataframe

In [ ]:
df_spacing_ik_bench = df_spacing_ik.copy()

bench_map = df_header.set_index("uwi")["bench"]

df_spacing_ik_bench["bench_i"] = df_spacing_ik_bench["well_i"].map(bench_map)
df_spacing_ik_bench["bench_k"] = df_spacing_ik_bench["well_k"].map(bench_map)

df_spacing_ik_bench = reorder_columns(df=df_spacing_ik_bench, columns_to_move=['bench_i', 'bench_k'], reference_column='well_k')
df_spacing_ik_bench = reorder_columns(df=df_spacing_ik_bench, 
                                        columns_to_move=['direction_to_k_from_i_axis','overlap_pct_i', 'overlap_pct_k','overlap_len_common_ft', 'LL_i', 'LL_k'], 
                                        reference_column='3D_dist')

### 5.2 Running oneline

In [24]:
# Filter out rows where 'reject_reason' is not empty
df_spacing_ik_bench_filt = df_spacing_ik_bench[df_spacing_ik_bench["reject_reason"]==""].reset_index(drop=True).copy()

In [25]:
nb = DirectionalBenchNeighbors()

# A) Classic: 1320′ cutoff, any axis, no preference
res_any = nb.summarize(spacing_df=df_spacing_ik_bench_filt, header_df=df_header, 
                       cutoff_ft=1800.0, vertical_cutoff_ft=150.00, 
                       overlap_pct_k_min=0.30)   # require ≥30% of k overlaps with i

# B) Only EW is eligible for *_1, and prefer EW if ties occur
res_ew = nb.summarize(spacing_df=df_spacing_ik_bench_filt, header_df=df_header, 
                      cutoff_ft=1800.0, vertical_cutoff_ft=150.00, overlap_pct_k_min=0.30, # require ≥30% of k overlaps with i
                      axis_mode="EW", prefer_axis="EW")

# C) Any axis is eligible for *_1, but prefer NS on distance ties
res_any_pref_ns = nb.summarize(spacing_df=df_spacing_ik_bench_filt, header_df=df_header, 
                               cutoff_ft=1800.0, vertical_cutoff_ft=150.00, overlap_pct_k_min=0.30, # require ≥30% of k overlaps with i
                               axis_mode="any", prefer_axis="NS")

In [26]:
res_ew

,well_i,uwi_same_1,hz_ft_to_same_1,vt_ft_to_same_1,3d_ft_to_same_1,uwi_same_2,hz_ft_to_same_2,vt_ft_to_same_2,3d_ft_to_same_2,uwi_near_1,hz_ft_to_near_1,vt_ft_to_near_1,3d_ft_to_near_1,uwi_near_2,hz_ft_to_near_2,vt_ft_to_near_2,3d_ft_to_near_2
0,30025410040100,30025503690000,933.507723,23.6125,933.806307,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,30025421210000,30025426220000,1107.525915,26.0445,1107.832103,30025428730000,1754.503370,46.6605,1755.123721,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,30025426220000,30025421210000,1107.539330,26.0445,1107.845514,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,30025428730000,30025421210000,1754.494781,46.6605,1755.115136,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,30025437530100,30025507280000,863.991208,63.7220,866.337868,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1840,42501376070000,42501376080000,715.778692,9.7000,715.844415,NaN,NaN,NaN,NaN,42501366580000,850.667092,31.890,851.264632,42501371140000,1466.575518,41.6600,1467.167102
1841,42501376080000,42501376070000,715.777839,9.7000,715.843562,42501376100000,824.248755,7.7850,824.285518,42501371140000,756.582689,51.360,758.323952,42501366580000,1572.672532,41.5900,1573.222369
1842,42501376090000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,42501366970000,906.587087,68.695,909.185981,42501372890000,1700.058754,31.0685,1700.342618
1843,42501376100000,42501376080000,824.177946,7.7850,824.214713,NaN,NaN,NaN,NaN,42501366270000,695.426828,41.965,696.691850,NaN,NaN,NaN,NaN


In [27]:
df_spacing_ik_bench_filt

,well_i,well_k,bench_i,bench_k,horizontal_dist,horizontal_dist_median,vertical_dist,3D_dist,direction_to_k_from_i_axis,overlap_pct_i,overlap_pct_k,overlap_len_common_ft,LL_i,LL_k,drill_direction_i,drill_direction_k,n_samples,dy_p5,angle_deg,pair_alignment,min_distance_ft,mean_windowed_ft,reject_reason,direction_axis,direction_axis_confidence,direction_axis_distribution,axis_forced
0,30025410040100,30025503690000,SAN ANDRES,SAN ANDRES,933.507723,940.956180,23.6125,933.806307,W,1.000000,0.791128,4193.025870,4193.025870,5300.057789,NS,NS,42.0,889.457666,1.111058,parallel_like,NaN,NaN,,EW,1.0,"E:0.00,W:1.00",True
1,30025410040100,42501372540000,SAN ANDRES,SAN ANDRES,1505.026059,1505.095305,27.3745,1505.274992,E,0.061805,0.051191,259.151954,4193.025870,5062.409197,NS,NS,3.0,1500.649478,1.592600,parallel_like,NaN,NaN,,EW,1.0,"E:1.00,W:0.00",True
2,30025421210000,30025426220000,SAN ANDRES,SAN ANDRES,1107.525915,1107.988013,26.0445,1107.832103,W,0.999413,0.996295,4303.157692,4305.683334,4319.159191,NS,NS,44.0,1039.241090,2.100121,parallel_like,NaN,NaN,,EW,1.0,"E:0.00,W:1.00",True
3,30025421210000,30025428730000,SAN ANDRES,SAN ANDRES,1754.503370,1754.693717,46.6605,1755.123721,E,0.977401,0.999952,4208.380994,4305.683334,4208.582903,NS,NS,43.0,1728.266897,0.431092,parallel_like,NaN,NaN,,EW,1.0,"E:1.00,W:0.00",True
4,30025426220000,30025421210000,SAN ANDRES,SAN ANDRES,1107.539330,1108.080156,26.0445,1107.845514,E,0.986746,0.989835,4261.914221,4319.159191,4305.683334,NS,NS,43.0,1024.977509,2.100121,parallel_like,NaN,NaN,,EW,1.0,"E:1.00,W:0.00",True
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
13174,42501376110000,42501369840000,SAN ANDRES,SAN ANDRES,1259.351399,1251.028002,1.9450,1259.352901,W,0.958968,0.653560,4953.338460,5165.280109,7579.011865,NS,NS,50.0,1097.886273,3.442921,parallel_like,NaN,NaN,,EW,1.0,"E:0.00,W:1.00",True
13175,42501376110000,42501373020000,SAN ANDRES,SAN ANDRES,2300.705091,2302.019506,80.7150,2302.120506,E,0.968905,0.999639,5004.663453,5165.280109,5006.472706,NS,NS,51.0,2277.366611,0.137303,parallel_like,NaN,NaN,,EW,1.0,"E:1.00,W:0.00",True
13176,42501376110000,42501373040000,SAN ANDRES,SAN ANDRES,668.991547,677.768394,61.9835,671.856863,E,0.964660,0.998868,4982.741352,5165.280109,4988.388101,NS,NS,50.0,546.092506,2.734059,parallel_like,NaN,NaN,,EW,1.0,"E:1.00,W:0.00",True
13177,42501376110000,42501373050000,SAN ANDRES,SAN ANDRES,834.861571,823.656159,46.1885,836.138278,W,0.978777,0.998863,5055.658011,5165.280109,5061.413897,NS,NS,51.0,713.693706,2.580199,parallel_like,NaN,NaN,,EW,1.0,"E:0.00,W:1.00",True
